# 1.1. LU-разложение

Реализация: `lab1/lu.py` (с выбором опорного элемента). Данные читаются из файлов, размерность матрицы не задана в коде.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

from lab1.dataio import read_matrix, read_vector, read_scalar, read_tridiag
from lab1.matops import (
    mat_mul, mat_vec, identity, fmt_mat, fmt_vec, mat_sub, max_abs_mat, diag_from, transpose,
)

from lab1.lu import lu_decompose, lu_solve, lu_inverse, lu_det

A = read_matrix(DATA / "1_1" / "A.txt")
b = read_vector(DATA / "1_1" / "b.txt")
print("A")
print(fmt_mat(A))
print("b =", fmt_vec(b))
print("n =", len(A))

A
             0              2              3              1
             4              5              6              2
             1              0              2              1
             3              1              4              5
b = [6, 17, 4, 13]
n = 4


## Матрицы L и U

In [2]:
L, U, P, swaps = lu_decompose(A)
print("число перестановок строк:", swaps)
print("L")
print(fmt_mat(L))
print()
print("U")
print(fmt_mat(U))
print()
print("P")
print(fmt_mat(P))

число перестановок строк: 3
L
             1              0              0              0
          0.75              1              0              0
             0    -0.72727273              1              0
          0.25     0.45454545     0.27586207              1

U
             4              5              6              2
             0          -2.75           -0.5            3.5
             0              0      2.6363636      3.5454545
             0              0              0     -2.0689655

P
             0              1              0              0
             0              0              0              1
             1              0              0              0
             0              0              1              0


## L · U

In [3]:
LU = mat_mul(L, U)
print("L · U")
print(fmt_mat(LU))

L · U
             4              5              6              2
             3              1              4              5
             0              2              3              1
             1              0              2              1


## Решение системы Ax = b

In [4]:
x = lu_solve(L, U, P, b)
print("x =", fmt_vec(x))
print("проверка A x =", fmt_vec(mat_vec(A, x)))
print("b         =", fmt_vec(b))

x = [1, 1, 1, 1]
проверка A x = [6, 17, 4, 13]
b         = [6, 17, 4, 13]


## Обратная матрица через LU

In [5]:
Ainv = lu_inverse(L, U, P)
print("A^{-1}")
print(fmt_mat(Ainv))

A^{-1}
   -0.53333333     0.21666667     0.18333333   -0.016666667
    0.13333333     0.13333333    -0.73333333    0.066666667
           0.2          -0.05           0.65          -0.15
    0.13333333    -0.11666667    -0.48333333     0.31666667


## Определитель через LU

In [6]:
detA = lu_det(U, swaps)
print("det(A) =", detA)

det(A) = -59.99999999999999


## Проверка A · A⁻¹ = E

In [7]:
prod = mat_mul(A, Ainv)
E = identity(len(A))
print("A · A^{-1}")
print(fmt_mat(prod))
print()
print("макс. |A A^{-1} - E| =", max_abs_mat(mat_sub(prod, E)))

A · A^{-1}
             1  2.7755576e-17              0 -5.5511151e-17
  1.110223e-16              1              0   1.110223e-16
-5.5511151e-17  2.7755576e-17              1  5.5511151e-17
 -2.220446e-16              0              0              1

макс. |A A^{-1} - E| = 2.220446049250313e-16


## Проверка L · U = P · A

In [8]:
PA = mat_mul(P, A)
print("P · A")
print(fmt_mat(PA))
print()
print("L · U")
print(fmt_mat(LU))
print()
print("макс. |L U - P A| =", max_abs_mat(mat_sub(LU, PA)))

P · A
             4              5              6              2
             3              1              4              5
             0              2              3              1
             1              0              2              1

L · U
             4              5              6              2
             3              1              4              5
             0              2              3              1
             1              0              2              1

макс. |L U - P A| = 0.0


## Сравнение с NumPy (только проверка)

In [9]:
import numpy as np

print("numpy x     =", np.linalg.solve(np.array(A), np.array(b)))
print("numpy det   =", np.linalg.det(np.array(A)))
print("numpy Ainv\n", np.linalg.inv(np.array(A)))

numpy x     = [1. 1. 1. 1.]
numpy det   = -60.000000000000036
numpy Ainv
 [[-0.53333333  0.21666667  0.18333333 -0.01666667]
 [ 0.13333333  0.13333333 -0.73333333  0.06666667]
 [ 0.2        -0.05        0.65       -0.15      ]
 [ 0.13333333 -0.11666667 -0.48333333  0.31666667]]
